# U2U/U2C comparison across five topology runs

The server scripts show five CelebA random-topology runs (rand_topology_00 ... _04). Their backbone is frozen while U2U and U2C are trained. The notebook uses the same 10 test images and their true concept labels in every run. It saves U2U vectors, logits, predicted concept scores, ground-truth concept vectors, topology masks, and effective U2C weights. Displays include the first image's 5?70 U matrix and, for each topology, predicted concept scores beside the true concept vector.


In [ ]:
from pathlib import Path
import json, sys
import numpy as np, pandas as pd, torch, yaml, matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from src.models import Template_CBM_MultiClass, UtoY_model
from src.utils import get_component_with_dicts

CONFIG_DIR=ROOT/'all_configs/sanity_checks/celeba_topology_random_full_no_side_channel'
EXPERIMENT_ROOT=ROOT/'experiments/CelebA/train_cbm/Standard_CelebA/topology_random_full_no_side_channel/CREAM_celeba_topology_random_full_no_side_channel'
RUNS=[]
for i in range(5):
    name=f'rand_topology_{i:02d}'
    RUNS.append({
        'name':name,
        'config':CONFIG_DIR/f'CREAM_celeba_topology_random_full_no_side_channel_{name}.yaml',
        'checkpoint_glob':EXPERIMENT_ROOT/name/'lightning_logs'/'version_*'/'checkpoints'/'*.ckpt',
    })
N_IMAGES=10
OUT_DIR=ROOT/'notebook'/'topology_u2c_debug_artifacts'; OUT_DIR.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
def load_run(spec):
    cp=Path(spec['config'])
    candidates=sorted(Path(ROOT).glob(str(Path(spec['checkpoint_glob']).relative_to(ROOT))),key=lambda p:p.stat().st_mtime)
    if not candidates: raise FileNotFoundError(f"No trained checkpoint found for {spec['name']} under {spec['checkpoint_glob']}")
    kp=candidates[-1]
    if not cp.is_file(): raise FileNotFoundError(f'Missing config: {cp}')
    cfg=yaml.safe_load(cp.read_text(encoding='utf-8')); h,paths=cfg['hyperparameters_model2'],cfg['paths']
    dag=torch.tensor(pd.read_csv(paths['DAG_file'],index_col=0).values,dtype=torch.bool)
    model_class=get_component_with_dicts('model',cfg['model_name'])
    backbone=model_class.load_from_checkpoint(checkpoint_path=ROOT/paths['input_model_path'],dataset=cfg['dataset_name'],frozen=True)
    groups=None
    if paths.get('softmax_mask'): groups=json.loads((ROOT/paths['softmax_mask']).read_text(encoding='utf-8'))
    u2c=UtoY_model(**h,causal_graph=dag,mutually_exclusive_concepts=groups)
    model=Template_CBM_MultiClass(backbone.concept_extractor,u2c,num_exogenous=h['num_exogenous'],num_classes=h['num_classes'],num_concepts=h['num_concepts'],num_side_channel=h.get('num_side_channel',0),frozen_model1=True,concept_representation=h.get('concept_representation','logits'),learning_rate=cfg.get('hyperparameters',{}).get('learning_rate',1e-4),lambda_weight=cfg.get('hyperparameters',{}).get('lambda_weight',1.0))
    payload=torch.load(kp,map_location='cpu',weights_only=False); state=payload.get('state_dict',payload)
    missing,unexpected=model.load_state_dict(state,strict=False)
    bad=[k for k in missing if k.startswith(('u_to_CY.','x_to_u.'))]
    if bad: raise RuntimeError(f'Checkpoint missing model weights: {bad[:10]}')
    model=model.to(DEVICE).eval()
    return cfg,model

assert len(RUNS)==5
loaded=[(spec,*load_run(spec)) for spec in RUNS]
# Build the test loader once. This guarantees identical image order for each model.
spec0,cfg0,_=loaded[0]
dataset_class=get_component_with_dicts('dataset',cfg0['dataset_name'])
dataset=dataset_class(**cfg0['dataset_params'])
dataset.setup('test')
test_loader=dataset.test_dataloader()
print('runs:',[r['name'] for r in RUNS], '| test split:',len(dataset.test_data))


In [ ]:
# Collect the same first 10 test images and their ground-truth concept vectors.
images, true_concepts = [], []
for batch in test_loader:
    images.extend(batch[0])
    true_concepts.extend(batch[1])
    if len(images)>=N_IMAGES: break
if len(images)<N_IMAGES: raise ValueError(f'Only found {len(images)} test images')
x=torch.stack(images[:N_IMAGES]).to(DEVICE)
TRUE_CONCEPTS=torch.stack(true_concepts[:N_IMAGES]).cpu().numpy().astype(float)
CONCEPT_NAMES=list(getattr(dataset,'concept_names',[f'c{i+1}' for i in range(TRUE_CONCEPTS.shape[1])]))
if TRUE_CONCEPTS.shape[1] != loaded[0][2].num_concepts:
    raise ValueError(f'True concept width {TRUE_CONCEPTS.shape[1]} does not match model concept count {loaded[0][2].num_concepts}')
Us,LOGITS,SCORES,WEIGHTS,MASKS=[],[],[],[],[]
with torch.inference_mode():
    for spec,cfg,model in loaded:
        backbone_features=model.x_to_u(x)
        u=model.u_to_CY.u2u_model(backbone_features)
        uc=u[:,:model.u_to_CY.num_exogenous-model.u_to_CY.num_side_channel]
        layer=next((m for m in model.u_to_CY.u2c_model.modules() if isinstance(m,torch.nn.Linear)),None)
        if layer is None: raise RuntimeError(f'No U2C linear layer in {spec["name"]}')
        raw=layer.weight.detach().cpu().numpy(); mask=getattr(layer,'mask',None)
        mask=mask.detach().cpu().numpy() if mask is not None else np.ones_like(raw)
        if mask.shape!=raw.shape and mask.T.shape==raw.shape: mask=mask.T
        logits=model.u_to_CY.u2c_model(uc); scores=model.u_to_CY.concept_activation_function(logits)
        Us.append(u.cpu().numpy()); LOGITS.append(logits.cpu().numpy()); SCORES.append(scores.cpu().numpy()); WEIGHTS.append(raw*mask); MASKS.append(mask)
U,LOGITS,SCORES,EFFECTIVE_W,MASKS=map(np.stack,(Us,LOGITS,SCORES,WEIGHTS,MASKS))
print('U [topology,image,U-dim]:',U.shape,'predicted concept scores:',SCORES.shape,'true concepts:',TRUE_CONCEPTS.shape,'effective U2C weights:',EFFECTIVE_W.shape)


In [ ]:
# Save numeric artifacts and readable CSVs.
np.savez_compressed(OUT_DIR/'topology_comparison.npz', U=U, logits=LOGITS, scores=SCORES, true_concepts=TRUE_CONCEPTS, effective_u2c_weights=EFFECTIVE_W, masks=MASKS)
names=[r['name'] for r in RUNS]
pd.DataFrame(U[:,0,:],index=names,columns=[f'u{i+1}' for i in range(U.shape[2])]).to_csv(OUT_DIR/'u_vectors_image_0.csv')
score_rows=[]
for t in range(5):
    for i in range(N_IMAGES):
        for c in range(SCORES.shape[2]):
            score_rows.append({'topology':names[t],'image_index':i,'concept_index':c,'concept_name':CONCEPT_NAMES[c],'logit':LOGITS[t,i,c],'predicted_score':SCORES[t,i,c],'true_concept':TRUE_CONCEPTS[i,c]})
scores_long=pd.DataFrame(score_rows)
scores_long.to_csv(OUT_DIR/'concept_scores_with_truth_5_topologies_10_images.csv',index=False)
pd.DataFrame(TRUE_CONCEPTS,index=[f'image_{i}' for i in range(N_IMAGES)],columns=CONCEPT_NAMES).to_csv(OUT_DIR/'true_concept_vectors_10_images.csv')
for t,name in enumerate(names): pd.DataFrame(EFFECTIVE_W[t],index=CONCEPT_NAMES,columns=[f'u{i+1}' for i in range(EFFECTIVE_W.shape[2])]).to_csv(OUT_DIR/f'{name}_effective_u2c_weights.csv')
(OUT_DIR/'metadata.json').write_text(json.dumps({'runs':[{'name':r['name'],'config':str(r['config']),'checkpoint_glob':str(r['checkpoint_glob'])} for r in RUNS],'concept_names':CONCEPT_NAMES,'U_shape':list(U.shape),'scores_shape':list(SCORES.shape),'true_concepts_shape':list(TRUE_CONCEPTS.shape),'effective_weights_shape':list(EFFECTIVE_W.shape),'images':N_IMAGES},indent=2))
print('Saved:',OUT_DIR)


In [ ]:
# U2U representation across topologies for the first selected image (5 ? 70 for this CelebA config).
fig,ax=plt.subplots(figsize=(max(12,U.shape[2]*.22),3.5)); im=ax.imshow(U[:,0,:],aspect='auto',cmap='coolwarm'); ax.set_yticks(range(5),names); ax.set_xlabel('U2U feature'); ax.set_title('U2U vectors for image 0'); fig.colorbar(im,ax=ax); plt.tight_layout()
# For each topology, show predicted concept scores beside that image's true labels.
image_idx=0
fig,axes=plt.subplots(5,1,figsize=(max(12,SCORES.shape[2]*.28),12),sharex=True)
for t,ax in enumerate(axes):
    ax.plot(range(len(CONCEPT_NAMES)),SCORES[t,image_idx],marker='o',label='predicted score')
    ax.scatter(range(len(CONCEPT_NAMES)),TRUE_CONCEPTS[image_idx],marker='x',s=55,label='true concept')
    ax.set_ylabel(names[t]); ax.grid(alpha=.2)
axes[0].legend(loc='upper right'); axes[-1].set_xticks(range(len(CONCEPT_NAMES)),CONCEPT_NAMES,rotation=35,ha='right')
fig.suptitle(f'Predicted vs true concept vector for test image {image_idx}'); plt.tight_layout()
# All 10 images: predicted scores by topology; ground truth is shown in a separate panel.
fig,axes=plt.subplots(1,6,figsize=(22,4),sharey=True)
for t,ax in enumerate(axes[:5]):
    ax.imshow(SCORES[t],aspect='auto',vmin=0,vmax=1,cmap='viridis'); ax.set_title(names[t]); ax.set_xlabel('Concept')
axes[5].imshow(TRUE_CONCEPTS,aspect='auto',vmin=0,vmax=1,cmap='viridis'); axes[5].set_title('Ground truth'); axes[5].set_xlabel('Concept')
axes[0].set_ylabel('Image index'); fig.suptitle('Concept scores for the same 10 images'); plt.tight_layout()
fig,axes=plt.subplots(1,5,figsize=(18,5),sharey=True)
for t,ax in enumerate(axes): ax.imshow(EFFECTIVE_W[t],aspect='auto',cmap='coolwarm'); ax.set_title(names[t]); ax.set_xlabel('U2C input feature')
axes[0].set_ylabel('Predicted concept'); fig.suptitle('Effective U2C weights: learned weights ? topology mask'); plt.tight_layout()
scores_long.head(20)
